# Gold Layer — Borough Priority Analysis

**Research question**: Which London boroughs have the worst social housing stock AND the most financially vulnerable tenants?

**Data sources**:
- EPC silver → housing stock quality by borough
- IMD 2019 (LSOA-level) → income/employment deprivation aggregated to borough
- CORE silver → London-wide affordability trends (no borough breakdown available)

## Business Context: Why EPC band matters for a housing association

### Regulatory pressure

**Minimum Energy Efficiency Standards (MEES)** already prohibit letting properties below band E. The government is legislating to raise this minimum to band C for social housing by 2030 and private rentals by 2028. A property that fails to reach band C by the deadline cannot legally be marketed for a new tenancy until improvements are made. For a housing association with tens of thousands of properties, this is an existential operational risk — not a compliance footnote.

**The Regulator of Social Housing** requires associations to report stock condition data. Borough-level EPC tracking (as produced in this pipeline) is the foundation of that reporting.

### Fuel poverty

The government’s fuel poverty metric — Low Income Low Energy Efficiency (LILEE) — classifies a household as fuel poor if they live in a property below band C AND have a low income. This directly links EPC band to tenant welfare. the housing association’s mission is explicitly tenant-focused: improving stock from D to C removes tenants from the fuel poverty definition entirely, reduces energy bills (typically £300–£700/year saving at band C vs D), and reduces health risk.

### Health consequences of poor stock

Properties rated F or G are associated with damp, mould, and cold indoor temperatures. Public Health England links cold homes to approximately 10,000 excess winter deaths per year in England. The social cost — NHS pressure, lost working days, mental health impact — falls disproportionately on social housing tenants who cannot afford to move or heat their homes adequately.

### The Warm Homes Fund opportunity

The £1.29bn Warm Homes: Social Housing Fund (2025–2028) provides competitive grants for exactly these improvements. To bid successfully, associations must demonstrate which properties are worst performing, which tenants are most vulnerable, and provide cost evidence. This pipeline produces all three: the borough priority ranking (worst stock + most deprived tenants), the retrofit cost scenarios (optimistic/central/pessimistic per home and total), and the improvement-type breakdown showing what works are needed and at what scale.

The analysis below produces that prioritisation.

In [1]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, sum as spark_sum, round as spark_round,
    when, lit, year as spark_year, to_date, desc, dense_rank
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('gold_layer') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

EPC_SILVER  = '../data/silver/epc'
CORE_SILVER = '../data/silver/core'
IMD_CSV     = '../data/bronze/imd/imd2019_lsoa.csv'
GOLD        = '../data/gold'

epc  = spark.read.parquet(EPC_SILVER)
core = spark.read.parquet(CORE_SILVER)
print(f'EPC: {epc.count():,} | CORE: {core.count():,}')

EPC: 612,357 | CORE: 501,244


## 1. Aggregate IMD to borough level

In [2]:
imd_raw = spark.read.csv(IMD_CSV, header=True, inferSchema=True)

# Filter to London (LA codes E09xxxxxxx), aggregate LSOA → borough
imd_borough = (
    imd_raw
    .filter(col('`Local Authority District code (2019)`').startswith('E09'))
    .groupBy(
        col('`Local Authority District name (2019)`').alias('la_name')
    )
    .agg(
        spark_round(avg('`Index of Multiple Deprivation (IMD) Score`'), 3).alias('imd_score'),
        spark_round(avg('`Income Score (rate)`'), 4).alias('income_deprivation_rate'),
        spark_round(avg('`Employment Score (rate)`'), 4).alias('employment_deprivation_rate'),
        spark_round(avg('`Health Deprivation and Disability Score`'), 3).alias('health_deprivation_score'),
        spark_round(avg('`Living Environment Score`'), 3).alias('living_env_score'),
        count('*').alias('lsoa_count')
    )
    .orderBy(desc('imd_score'))
)

print(f'London boroughs in IMD: {imd_borough.count():,}')
display(imd_borough.limit(10).toPandas().style.format(thousands=","))

London boroughs in IMD: 33
+--------------------+---------+-----------------------+---------------------------+------------------------+----------------+----------+
|la_name             |imd_score|income_deprivation_rate|employment_deprivation_rate|health_deprivation_score|living_env_score|lsoa_count|
+--------------------+---------+-----------------------+---------------------------+------------------------+----------------+----------+
|Barking and Dagenham|32.883   |0.1942                 |0.119                      |0.223                   |28.924          |110       |
|Hackney             |32.868   |0.1985                 |0.119                      |0.394                   |35.496          |144       |
|Newham              |29.771   |0.1715                 |0.0964                     |-0.011                  |31.846          |164       |
|Tower Hamlets       |27.97    |0.1907                 |0.1047                     |0.21                    |33.236          |144       |
|Isling

## 2. Aggregate EPC to borough level

In [3]:
epc_borough = (
    epc
    .groupBy('borough')
    .agg(
        count('*').alias('social_properties'),
        spark_round(avg('epc_score'), 1).alias('avg_epc_score'),
        spark_round(
            spark_sum(when(col('below_epc_c') == True, 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_below_epc_c'),
        spark_round(
            spark_sum(when(col('construction_age_band').isin(
                'England and Wales: before 1900',
                'England and Wales: 1900-1929',
                'England and Wales: 1930-1949'
            ), 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_pre_1950'),
        spark_round(avg('co2_emissions'), 2).alias('avg_co2_per_m2'),
    )
)

print(f'Boroughs in EPC: {epc_borough.count():,}')
display(epc_borough.orderBy(desc('pct_below_epc_c')).limit(10).toPandas().style.format(thousands=","))

Boroughs in EPC: 34
+----------------------+-----------------+-------------+---------------+------------+--------------+
|borough               |social_properties|avg_epc_score|pct_below_epc_c|pct_pre_1950|avg_co2_per_m2|
+----------------------+-----------------+-------------+---------------+------------+--------------+
|Barking and Dagenham  |15346            |66.3         |56.0           |44.0        |2.75          |
|Enfield               |22532            |65.1         |52.4           |29.1        |2.86          |
|Redbridge             |8859             |67.2         |50.9           |26.7        |2.68          |
|Barnet                |22545            |67.0         |50.3           |39.2        |2.75          |
|Haringey              |20370            |67.3         |48.5           |47.4        |2.56          |
|Lambeth               |42811            |67.3         |46.9           |46.0        |2.56          |
|Camden                |20876            |67.6         |46.8           

## 3. Join EPC + IMD on borough name

EPC uses full borough names (e.g. 'City of London'). IMD uses the same. Join on name after trimming.

In [4]:
from pyspark.sql.functions import trim, upper, regexp_replace

# Normalise names for join
epc_norm = epc_borough.withColumn('borough_key', upper(trim(col('borough'))))
imd_norm = imd_borough.withColumn('borough_key', upper(trim(col('la_name'))))

# Check what EPC borough names look like
print('EPC borough names (sample):')
display(epc_norm.select('borough', 'borough_key').limit(5).toPandas().style.format(thousands=","))

print('IMD LA names (London, sample):')
display(imd_norm.select('la_name', 'borough_key').limit(5).toPandas().style.format(thousands=","))

EPC borough names (sample):
+---------+-----------+
|borough  |borough_key|
+---------+-----------+
|Lambeth  |LAMBETH    |
|Barnet   |BARNET     |
|Lewisham |LEWISHAM   |
|Islington|ISLINGTON  |
|Bexley   |BEXLEY     |
+---------+-----------+
only showing top 5 rows
IMD LA names (London, sample):
+--------------------+--------------------+
|la_name             |borough_key         |
+--------------------+--------------------+
|Barking and Dagenham|BARKING AND DAGENHAM|
|Hackney             |HACKNEY             |
|Newham              |NEWHAM              |
|Tower Hamlets       |TOWER HAMLETS       |
|Islington           |ISLINGTON           |
+--------------------+--------------------+
only showing top 5 rows


In [5]:
joined = epc_norm.join(imd_norm, on='borough_key', how='inner').drop('borough_key', 'la_name')

print(f'Boroughs matched: {joined.count():,} / 33')

# Show any EPC boroughs that didn't match IMD
unmatched = epc_norm.join(imd_norm, on='borough_key', how='left_anti')
if unmatched.count() > 0:
    print('Unmatched EPC boroughs:')
    unmatched.select('borough').show(truncate=False)

Boroughs matched: 33 / 33
Unmatched EPC boroughs:
+-------+
|borough|
+-------+
|NULL   |
+-------+



## 4. Compute composite priority score

Rank each borough on:
- % social stock below EPC C (housing quality)
- IMD income deprivation rate (financial vulnerability)
- % pre-1950 stock (future retrofit cost)

Composite = average rank across three dimensions. Rank 1 = most in need.

In [6]:
w_epc    = Window.orderBy(desc('pct_below_epc_c'))
w_income = Window.orderBy(desc('income_deprivation_rate'))
w_age    = Window.orderBy(desc('pct_pre_1950'))
w_final  = Window.orderBy('priority_score')

priority = (
    joined
    .withColumn('rank_epc',    dense_rank().over(w_epc))
    .withColumn('rank_income', dense_rank().over(w_income))
    .withColumn('rank_age',    dense_rank().over(w_age))
    .withColumn('priority_score',
        spark_round((col('rank_epc') + col('rank_income') + col('rank_age')) / 3.0, 1)
    )
    .withColumn('priority_rank', dense_rank().over(w_final))
    .select(
        'priority_rank', 'borough', 'priority_score',
        'pct_below_epc_c', 'income_deprivation_rate', 'pct_pre_1950',
        'avg_epc_score', 'imd_score', 'avg_co2_per_m2', 'social_properties',
        'rank_epc', 'rank_income', 'rank_age'
    )
    .orderBy('priority_rank')
)

print('=== BOROUGH PRIORITY RANKING ===')
print('(Rank 1 = worst housing stock + most deprived tenants)')
display(priority.limit(33).toPandas().style.format(thousands=","))

=== BOROUGH PRIORITY RANKING ===
(Rank 1 = worst housing stock + most deprived tenants)
+-------------+----------------------+--------------+---------------+-----------------------+------------+-------------+---------+--------------+-----------------+--------+-----------+--------+
|priority_rank|borough               |priority_score|pct_below_epc_c|income_deprivation_rate|pct_pre_1950|avg_epc_score|imd_score|avg_co2_per_m2|social_properties|rank_epc|rank_income|rank_age|
+-------------+----------------------+--------------+---------------+-----------------------+------------+-------------+---------+--------------+-----------------+--------+-----------+--------+
|1            |Barking and Dagenham  |3.0           |56.0           |0.1942                 |44.0        |66.3         |32.883   |2.75          |15346            |1       |2          |6       |
|2            |Haringey              |5.0           |48.5           |0.1683                 |47.4        |67.3         |27.587   |2.56  

In [7]:
priority.write.mode('overwrite').parquet(f'{GOLD}/borough_priority')
print('Saved borough_priority')

Saved borough_priority


## 5. London-wide affordability trend (CORE)

No borough breakdown in CORE, but shows how rent burden has changed 2007–2022.

In [8]:
affordability = (
    core
    .filter(col('weekly_rent_est').isNotNull() & col('weekly_income_est').isNotNull())
    .groupBy('year')
    .agg(
        count('*').alias('lettings'),
        spark_round(avg('weekly_rent_est'), 2).alias('avg_weekly_rent_£'),
        spark_round(avg('weekly_income_est'), 2).alias('avg_weekly_income_£'),
        spark_round(avg('rent_to_income_pct'), 1).alias('avg_rent_to_income_%'),
        spark_round(
            spark_sum(when(col('rent_to_income_pct') > 50, 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_spending_over_50pct_on_rent'),
        spark_round(
            spark_sum(when(col('overcrowded') == True, 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_overcrowded'),
    )
    .orderBy('year')
)

print('London affordability trend:')
display(affordability.limit(20).toPandas().style.format(thousands=","))

affordability.write.mode('overwrite').parquet(f'{GOLD}/london_affordability_trend')
print('Saved london_affordability_trend')

London affordability trend:
+----+--------+-----------------+-------------------+--------------------+-------------------------------+---------------+
|year|lettings|avg_weekly_rent_£|avg_weekly_income_£|avg_rent_to_income_%|pct_spending_over_50pct_on_rent|pct_overcrowded|
+----+--------+-----------------+-------------------+--------------------+-------------------------------+---------------+
|2007|    4595|            72.24|             228.24|                39.1|                           26.1|            0.0|
|2008|    7140|            74.05|             233.47|                39.3|                           27.9|            0.0|
|2009|    8284|             77.1|              238.6|                39.8|                           29.0|            0.0|
|2010|    7621|            77.89|             241.99|                40.1|                           29.0|            0.0|
|2011|   12263|            95.65|             228.53|                53.6|                           47.1|     

## 6. EPC improvement trend by borough

Have the worst boroughs been improving?

In [9]:
# Top 5 worst boroughs from priority ranking
top5 = [row['borough'] for row in priority.limit(5).select('borough').collect()]
print(f'Top 5 priority boroughs: {top5}')

epc_trend = (
    epc
    .withColumn('inspection_year', spark_year(to_date(col('inspection_date'), 'yyyy-MM-dd')))
    .filter(col('inspection_year').between(2012, 2024))
    .filter(col('borough').isin(top5))
    .groupBy('borough', 'inspection_year')
    .agg(
        count('*').alias('inspections'),
        spark_round(avg('epc_score'), 1).alias('avg_epc_score'),
        spark_round(
            spark_sum(when(col('below_epc_c') == True, 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_below_epc_c'),
    )
    .orderBy('borough', 'inspection_year')
)

display(epc_trend.limit(60).toPandas().style.format(thousands=","))
epc_trend.write.mode('overwrite').parquet(f'{GOLD}/epc_trend_top_boroughs')
print('Saved epc_trend_top_boroughs')

Top 5 priority boroughs: ['Barking and Dagenham', 'Haringey', 'Lambeth', 'Hammersmith and Fulham', 'Enfield']
+----------------------+---------------+-----------+-------------+---------------+
|borough               |inspection_year|inspections|avg_epc_score|pct_below_epc_c|
+----------------------+---------------+-----------+-------------+---------------+
|Barking and Dagenham  |2012           |397        |66.9         |52.4           |
|Barking and Dagenham  |2013           |1870       |66.9         |53.2           |
|Barking and Dagenham  |2014           |1482       |64.3         |67.3           |
|Barking and Dagenham  |2015           |2426       |65.8         |56.7           |
|Barking and Dagenham  |2016           |964        |66.2         |56.0           |
|Barking and Dagenham  |2017           |750        |65.0         |63.6           |
|Barking and Dagenham  |2018           |1100       |65.8         |62.5           |
|Barking and Dagenham  |2019           |830        |66.8    

## 7. Final answer

In [ ]:
print('=' * 65)
print('ANSWER: Which boroughs need most attention?')
print('(worst housing stock + most deprived tenants)')
print('=' * 65)
display(spark.read.parquet(f'{GOLD}/borough_priority') \
     .select('priority_rank','borough','pct_below_epc_c','income_deprivation_rate','pct_pre_1950') \
     .limit(10).toPandas().style.format(thousands=","))

print('=' * 65)
print('CONTEXT: London-wide rent burden (most recent year)')
print('=' * 65)
spark.read.parquet(f'{GOLD}/london_affordability_trend') \
     .orderBy(desc('year')).limit(3).show(truncate=False)